# Results

Charts for the team round robins, the fair tests and the ladder. Each section reads whatever result files are on disk, so re-run it after copying new results.

**Getting the data from ilab** (laptop, from the repo root):

```
rsync -a gpu:ai-vgc/data/team_rr gpu:ai-vgc/data/team_tests data/ && rsync -a --include='vgc-nn-team-*.out' --exclude='*' gpu:ai-vgc/logs/ logs/
```

Ladder games are already on the laptop (`data/live_games/`).

**Running it:** `uv sync --extra nn --extra viz`, then `uv run --extra viz jupyter lab notebooks/results.ipynb`.

In [ ]:
import json, math, re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RR, TESTS, LOGS, LIVE = ROOT / "data/team_rr", ROOT / "data/team_tests", ROOT / "logs", ROOT / "data/live_games"
pd.set_option("display.width", 160, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def wilson(w, n, z=1.96):
    """95% interval for w wins out of n."""
    if n == 0:
        return (np.nan, np.nan)
    p = w / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

## Round robins

Each job in `data/team_rr/<job>/` is one round robin: one CSV row per pair of entries. An entry is a team, or `TEAM@TAG` when the same team is entered under several pilots (v6/v7/v8).

In [ ]:
def load_rr(job):
    rows = pd.concat([pd.read_csv(p) for p in sorted((RR / str(job)).glob("[0-9]*.csv"))], ignore_index=True)
    return rows


def rr_jobs():
    out = []
    for d in sorted(RR.glob("*"), key=lambda p: p.stat().st_mtime):
        if not d.is_dir() or not any(d.glob("[0-9]*.csv")):
            continue
        r = load_rr(d.name)
        entries = sorted(set(r.team_a) | set(r.team_b))
        tags = sorted({e.split("@")[1] for e in entries if "@" in e})
        out.append({"job": d.name, "entries": len(entries), "pairs": len(r),
                    "series per pair": int(r.series.median()), "pilots": " ".join(tags) or "-",
                    "finished": pd.Timestamp(d.stat().st_mtime, unit="s").strftime("%m-%d %H:%M")})
    return pd.DataFrame(out)


jobs = rr_jobs() if RR.exists() else pd.DataFrame()
jobs

In [ ]:
# Pick a round robin: the newest one, or set JOB = "409713" etc. from the table above.
JOB = jobs.job.iloc[-1] if len(jobs) else None


def rr_table(r):
    """Per entry: series won and played, win rate and its 95% interval."""
    t = pd.concat([r[["team_a", "series_a", "series"]].set_axis(["team", "won", "n"], axis=1),
                   r.assign(won=r.series - r.series_a)[["team_b", "won", "series"]].set_axis(["team", "won", "n"], axis=1)])
    t = t.groupby("team").sum()
    t["rate"] = t.won / t.n
    t[["lo", "hi"]] = [wilson(w, n) for w, n in zip(t.won, t.n)]
    return t.sort_values("rate", ascending=False)


if JOB:
    rr = load_rr(JOB)
    table = rr_table(rr)
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(table) + 1))
    y = np.arange(len(table))[::-1]
    colors = ["#4c78a8" if "@" not in t else {"v6": "#9d9d9d", "v7": "#4c78a8", "v8": "#f58518"}.get(t.split("@")[1], "#54a24b")
              for t in table.index]
    ax.barh(y, table.rate * 100, xerr=[(table.rate - table.lo) * 100, (table.hi - table.rate) * 100],
            color=colors, error_kw={"lw": 1, "capsize": 2})
    ax.set_yticks(y, table.index)
    ax.axvline(50, color="k", lw=0.8, ls=":")
    ax.set_xlabel("series won (%), with 95% interval")
    ax.set_title(f"Round robin {JOB}: {len(rr)} pairs")
    plt.tight_layout()
    display(table.assign(rate=(table.rate * 100).round(1), lo=(table.lo * 100).round(1), hi=(table.hi * 100).round(1)))

### Head to head

Row's series win rate against column. Each cell is one pair (often 100 series, so about ±10 points); the overall rate above is far more reliable.

In [ ]:
if JOB:
    order = list(table.index)
    m = pd.DataFrame(np.nan, index=order, columns=order)
    for a, b, sa, n in rr[["team_a", "team_b", "series_a", "series"]].itertuples(index=False):
        m.loc[a, b], m.loc[b, a] = sa / n, 1 - sa / n
    fig, ax = plt.subplots(figsize=(0.45 * len(order) + 2, 0.4 * len(order) + 1.5))
    im = ax.imshow(m.values * 100, cmap="RdBu", vmin=0, vmax=100)
    ax.set_xticks(range(len(order)), order, rotation=90)
    ax.set_yticks(range(len(order)), order)
    if len(order) <= 20:
        for i in range(len(order)):
            for j in range(len(order)):
                if not np.isnan(m.iat[i, j]):
                    ax.text(j, i, f"{m.iat[i, j] * 100:.0f}", ha="center", va="center", fontsize=7,
                            color="w" if abs(m.iat[i, j] - 0.5) > 0.3 else "k")
    fig.colorbar(im, ax=ax, shrink=0.7, label="row wins (%)")
    ax.set_title(f"Round robin {JOB}: head to head")
    plt.tight_layout()

### Pilot against pilot

Only for round robins with several pilots (`GEN="v6 v7"`, `GEN="v7 v8"`). **Same team** compares each team's two models directly; it's the cleanest test of which model plays better.

In [ ]:
if JOB:
    tags = sorted({t.split("@")[1] for t in table.index if "@" in t})
    if len(tags) == 2:
        x, y = tags
        same = rr[rr.team_a.str.split("@").str[0] == rr.team_b.str.split("@").str[0]].copy()
        same["team"] = same.team_a.str.split("@").str[0]
        # share of series won by pilot y in each mirror
        same["y_rate"] = np.where(same.team_a.str.endswith("@" + y), same.series_a, same.series - same.series_a) / same.series
        same = same.sort_values("y_rate")
        lo_hi = [wilson(round(r * n), n) for r, n in zip(same.y_rate, same.series)]
        fig, ax = plt.subplots(figsize=(7, 0.32 * len(same) + 1))
        yy = np.arange(len(same))
        ax.barh(yy, same.y_rate * 100, color="#f58518",
                xerr=[[(r - l) * 100 for r, (l, h) in zip(same.y_rate, lo_hi)],
                      [(h - r) * 100 for r, (l, h) in zip(same.y_rate, lo_hi)]], error_kw={"lw": 1, "capsize": 2})
        ax.set_yticks(yy, same.team)
        ax.axvline(50, color="k", lw=0.8, ls=":")
        ax.set_xlabel(f"{y} wins against {x} on the same team (%)")
        ax.set_title(f"{y} vs {x}, mirrors: {y} won {same.y_rate.mean() * 100:.1f}% on average")
        plt.tight_layout()
    else:
        print("one pilot per team in this round robin")

## Fair tests

`team_test` runs: one team, one model, against every pool team. The setup (team, model, opponent model, closed sheets) is read from each job's log in `logs/vgc-nn-team-test-<job>.out`. Logs written before 2026-09-30 don't include the model, so those runs show as `?`.

In [ ]:
def test_setup(job):
    log = LOGS / f"vgc-nn-team-test-{job}.out"
    line = next((l for l in log.read_text().splitlines() if "reg_m" in l and ".txt" in l), "") if log.exists() else ""
    arg = lambda k: (re.search(rf"--{k} (\S+)", line) or [None, None])[1]
    team = re.search(r"(\S+)\.txt", line)
    return {"team": Path(team[1]).name if team else "?",
            "model": Path(arg("model")).stem if arg("model") else "?",
            "opponent": Path(arg("opponent-model")).stem if arg("opponent-model") else "same as model",
            "sheets": "closed" if "--closed-sheets" in line else "open",
            "format": arg("format") or "?"}


def load_tests():
    rows = []
    for d in sorted(TESTS.glob("*")):
        csvs = sorted(d.glob("*.csv")) if d.is_dir() else []
        if not csvs:
            continue
        t = pd.concat([pd.read_csv(p) for p in csvs])
        w, n = int(t.wins.sum()), int(t.games.sum())
        lo, hi = wilson(w, n)
        rows.append({"job": d.name, **test_setup(d.name), "wins": w, "games": n, "opponent teams": len(t),
                     "rate": round(100 * w / n, 1), "lo": round(100 * lo, 1), "hi": round(100 * hi, 1)})
    return pd.DataFrame(rows)


tests = load_tests() if TESTS.exists() else pd.DataFrame()
tests

In [ ]:
# The closed-sheet fair test (opponents piloted by the human stand-in). Change these to see other test sets.
FAIR = dict(sheets="closed", opponent="all-bo3-bc-v2")

if len(tests):
    fair = tests[(tests.sheets == FAIR["sheets"]) & (tests.opponent == FAIR["opponent"])].copy()
    fair["pilot"] = fair.model.str.extract(r"rnad-(v\d+)")[0].fillna(fair.model)
    fair = fair.drop_duplicates(["team", "pilot"], keep="last")
    piv = fair.pivot(index="team", columns="pilot", values="rate")
    lo = fair.pivot(index="team", columns="pilot", values="lo")
    hi = fair.pivot(index="team", columns="pilot", values="hi")
    piv = piv.loc[piv.max(axis=1).sort_values(ascending=False).index]
    pilots = sorted(piv.columns)
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(piv)), 4))
    width = 0.8 / len(pilots)
    colors = {"v6": "#9d9d9d", "v7": "#4c78a8", "v8": "#f58518"}
    for k, p in enumerate(pilots):
        x = np.arange(len(piv)) + (k - (len(pilots) - 1) / 2) * width
        v = piv[p]
        ax.bar(x, v, width, label=p, color=colors.get(p),
               yerr=[(v - lo.loc[piv.index, p]).fillna(0), (hi.loc[piv.index, p] - v).fillna(0)],
               error_kw={"lw": 1, "capsize": 2})
    ax.set_xticks(range(len(piv)), piv.index)
    ax.set_ylim(max(0, np.nanmin(piv.values) - 8), 100)
    ax.set_ylabel("games won (%)")
    ax.set_title(f"Fair test: whole pool, {FAIR['sheets']} sheets, opponents {FAIR['opponent']}")
    ax.legend()
    plt.tight_layout()
    display(piv.round(1))

## Ladder

Rated games from `data/live_games/*/games.jsonl`. Each game is labelled by its **team** (the team file, recorded since 2026-09-29; older games by their six Pokémon) and **pilot** (the model, recorded since 2026-10-01; earlier games show `general`, the shared model of the time). A **session** is a run of games with the same team and pilot and no gap over 45 minutes.

Ratings: `before` is ours going into the game (from the player line), `rating` ours after it (the next game's `before`, since saved logs stop before Showdown's rating-change line), `opp rating` the opponent's. `expected` is the Elo win chance from the two ratings, so `won − expected` says how much better than its rating the bot played.

In [ ]:
GAP = pd.Timedelta(minutes=45)


def load_ladder():
    rows = []
    for f in sorted(LIVE.glob("*/games.jsonl")):
        for line in f.open():
            g = json.loads(line)
            log = g["log"]
            if "|rated|" not in log:
                continue
            me = g.get("user") or "nimnimbot"
            change = re.search(rf"\|raw\|{re.escape(me)}'s rating: (\d+) &rarr; <strong>(\d+)", log)
            mine = re.search(rf"\|player\|p\d\|{re.escape(me)}\|[^|]*\|(\d+)", log)
            theirs = re.search(rf"\|player\|p\d\|(?!{re.escape(me)}\|)[^|]+\|[^|]*\|(\d+)", log)
            game = re.search(r"<strong>Game (\d)</strong>", log)
            model = g.get("model")
            rows.append({
                "time": pd.Timestamp(g["time"]), "format": g["format"].replace("gen9championsvgc2026", ""),
                "won": g["result"] == "win",
                "before": int(change[1]) if change else (int(mine[1]) if mine else np.nan),
                "rating": int(change[2]) if change else np.nan,
                "opp rating": int(theirs[1]) if theirs else np.nan,
                "game": int(game[1]) if game else 1, "turns": g["turns"],
                "forfeit": "forfeited" in log and g["result"] == "win",
                "team": g.get("team") or " / ".join(sorted(g["ours"])),
                "pilot": re.sub(r"^mc-cts-rnad-|-MC\d+$", "", model) if model else "general",
            })
    d = pd.DataFrame(rows).sort_values("time").reset_index(drop=True)
    # Saved logs end before Showdown's rating-change line, so a game's rating after is the next
    # game's rating going in (per format; the latest game has none yet).
    d["rating"] = d["rating"].fillna(d.groupby("format")["before"].shift(-1))
    d["label"] = d.team + " · " + d.pilot
    d["expected"] = 1 / (1 + 10 ** ((d["opp rating"] - d["before"]) / 400))
    # A session: one bot run, i.e. the same pilot and format with no gap over GAP. With --use or
    # --top it rotates teams inside a session, so teams are told apart per game, not per session.
    new = (d.pilot != d.pilot.shift()) | (d.format != d.format.shift()) | (d.time.diff() > GAP)
    d["session"] = new.cumsum()
    return d


ladder = load_ladder() if LIVE.exists() else pd.DataFrame()
FORMAT = "regmc"  # closed-sheet Bo1; "regmcbo3" for the Bo3 ladder
lad = ladder[ladder.format == FORMAT]
print(f"{len(ladder)} rated games, {len(lad)} in {FORMAT}, {lad.session.nunique()} sessions")

### Rating by team and session

One grey line per session (a bot run), each game a dot coloured by its team; marker shape by pilot. Gaps between sessions are left empty.

In [ ]:
if len(lad):
    labels = list(dict.fromkeys(lad.label))
    teams = list(dict.fromkeys(lad.team))
    palette = plt.get_cmap("tab10")
    tcolor = {t: palette(i % 10) for i, t in enumerate(teams)}
    styles = ["-", "--", ":", "-."]
    pstyle = {p: styles[i % 4] for i, p in enumerate(dict.fromkeys(lad.pilot))}
    fig, ax = plt.subplots(figsize=(11, 4.5))
    marks = ["o", "s", "^", "D", "v"]
    pmark = {p: marks[i % 5] for i, p in enumerate(dict.fromkeys(lad.pilot))}
    r = lad.dropna(subset=["rating"])
    for s, d in r.groupby("session"):
        ax.plot(d.time, d.rating, color="0.75", lw=0.8, zorder=1)
    for lab, d in r.groupby("label", sort=False):
        ax.scatter(d.time, d.rating, s=10, marker=pmark[d.pilot.iloc[0]], color=tcolor[d.team.iloc[0]],
                   label=f"{lab} ({len(d)})", zorder=2)
    peak = lad.loc[lad.rating.idxmax()]
    ax.annotate(f"peak {int(peak.rating)}\n{peak.label}", (peak.time, peak.rating), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
    ax.set_ylabel("rating after game")
    ax.set_title(f"Ladder rating ({FORMAT}) by team and pilot")
    ax.legend(fontsize=8, loc="best")
    fig.autofmt_xdate()
    plt.tight_layout()

### Sessions

Each session's start and end rating, record and how it played against its rating.

In [ ]:
if len(lad):
    ses = lad.groupby("session").agg(
        label=("label", lambda x: " + ".join(dict.fromkeys(x.str.split(" · ").str[0])) + " · " + x.iloc[0].split(" · ")[1]), start=("time", "min"), end=("time", "max"), games=("won", "size"),
        won=("won", "sum"), rating_in=("before", "first"), rating_out=("rating", "last"),
        peak=("rating", "max"), opp=("opp rating", "mean"), over_expected=("expected", lambda e: np.nan))
    ses["over_expected"] = lad.groupby("session").apply(lambda d: (d.won - d.expected).mean() * 100)
    ses["change"] = ses.rating_out - ses.rating_in
    ses["win %"] = (100 * ses.won / ses.games).round(1)
    ses = ses[ses.games >= 3]
    fig, ax = plt.subplots(figsize=(11, 3.2))
    colors = [tcolor.get(l.split(" · ")[0], "0.5") for l in ses.label]
    ax.bar(range(len(ses)), ses.change, color=colors)
    ax.set_xticks(range(len(ses)), [f"{l}\n{s:%m-%d %H:%M}" for l, s in zip(ses.label, ses.start)], rotation=60,
                  ha="right", fontsize=7)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("rating change")
    ax.set_title("Rating change per session (3+ games)")
    plt.tight_layout()
    display(ses.assign(start=ses.start.dt.strftime("%m-%d %H:%M"), end=ses.end.dt.strftime("%H:%M"),
                       opp=ses.opp.round(0), over_expected=ses.over_expected.round(1))
            [["label", "start", "end", "games", "won", "win %", "rating_in", "rating_out", "change", "peak",
              "opp", "over_expected"]])

### Team summary

Per team and pilot. `win %` has a 95% interval; `over expected` is how many points above its Elo-expected win rate it played (0 = exactly its rating; ladder pairs push everyone towards 0 over time, so a positive number means it's still climbing). `forfeit wins` are wins where the opponent forfeited.

In [ ]:
def team_summary(d):
    g = d.groupby("label")
    s = pd.DataFrame({
        "games": g.size(), "won": g.won.sum(), "sessions": g.session.nunique(),
        "peak": g.rating.max(), "last": g.rating.last(), "mean rating": g.rating.mean().round(0),
        "mean opp": g["opp rating"].mean().round(0),
        "over expected": (g.apply(lambda x: (x.won - x.expected).mean()) * 100).round(1),
        "forfeit wins": g.forfeit.sum(), "turns": g.turns.mean().round(1),
        "first": g.time.min().dt.strftime("%m-%d"), "latest": g.time.max().dt.strftime("%m-%d %H:%M")})
    s["win %"] = (100 * s.won / s.games).round(1)
    ci = [wilson(w, n) for w, n in zip(s.won, s.games)]
    s["95% CI"] = [f"[{100 * a:.0f}, {100 * b:.0f}]" for a, b in ci]
    s = s.sort_values("peak", ascending=False)
    return s[["games", "won", "win %", "95% CI", "peak", "last", "mean rating", "mean opp", "over expected",
              "forfeit wins", "turns", "sessions", "first", "latest"]]


if len(lad):
    summary = team_summary(lad)
    display(summary)

In [ ]:
# Win rate by opponent rating, per team (teams with 30+ games)
if len(lad):
    big = lad.groupby("label").filter(lambda d: len(d) >= 30).dropna(subset=["opp rating"]).copy()
    big["bin"] = (big["opp rating"] // 100 * 100).astype(int)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for lab, d in big.groupby("label"):
        b = d.groupby("bin").won.agg(["mean", "size"])
        b = b[b["size"] >= 5]
        ax.plot(b.index, b["mean"] * 100, marker="o", label=f"{lab} ({len(d)})", color=tcolor[lab.split(" · ")[0]])
    ax.axhline(50, color="k", lw=0.8, ls=":")
    ax.set_xlabel("opponent rating (bins of 100, 5+ games)")
    ax.set_ylabel("won (%)")
    ax.set_title("Win rate by opponent rating")
    ax.legend(fontsize=8)
    plt.tight_layout()

In [ ]:
# Bo3: win rate by game in the series; everything: win rate by opponent rating.
if len(ladder):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    bo3 = ladder[ladder.format.str.endswith("bo3")]
    if len(bo3):
        g = bo3.groupby("game").won.agg(["mean", "size"])
        axes[0].bar(g.index.astype(str), g["mean"] * 100, color="#4c78a8")
        for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
            axes[0].text(i, m * 100 + 1, f"n={n}", ha="center", fontsize=8)
        axes[0].set_title("Bo3: win rate by game")
        axes[0].set_ylabel("won (%)")
        axes[0].axhline(50, color="k", lw=0.8, ls=":")
    d = ladder.dropna(subset=["opp rating"]).copy()
    d["bin"] = (d["opp rating"] // 100 * 100).astype(int)
    for fmt, x in d.groupby("format"):
        b = x.groupby("bin").won.agg(["mean", "size"])
        b = b[b["size"] >= 5]
        axes[1].plot(b.index, b["mean"] * 100, marker="o", label=fmt)
    axes[1].axhline(50, color="k", lw=0.8, ls=":")
    axes[1].set_xlabel("opponent rating (bins of 100, at least 5 games)")
    axes[1].set_title("Win rate by opponent rating")
    axes[1].legend()
    plt.tight_layout()